### Getting Started with Local AI

Let's get your computer set up to test out product ideas without needing any API keys. :) There are plenty of freely available local Ai models that can help you develop minimal MVPs without paying for a single token.

1. Get [ollama](https://ollama.com/) set up and running. You might also want to [download the desktop-based software](https://ollama.com/download) in case you want to test out prompts outside of these notebooks.

2. Get at least one [llamafile](https://github.com/Mozilla-Ocho/llamafile) running.

3. Set up a workflow for loading and saving prompts and responses.

In [1]:
import ollama

If you are running in a devcontainer, ollama is already running in another docker. That means you can use the next line to set up a client.  

In [6]:
# if you are running in a devcontiner, run this. If not, run the next line.
client = ollama.Client('http://ollama:11434')

**Do not** run the following line if you are in a devcontainer. This next line is for ppl who are running the Jupyter natively and have installed ollama python library on their system.

In [8]:
# only execute this code if you are NOT in a devcontainer :) 
client = ollama

In [9]:
for model in client.list().models:
    print(model.model)

llama-guard3:latest
mistral:latest
deepseek-r1:latest
qwen3-coder-next:latest
gemma3:1b
tinyllama:latest
qwen3:4b
gemma3:12b
qwen2.5vl:latest
mistral-small3.1:latest
deepseek-r1:8b
qwen3:8b
gemma3:27b
gpt-oss:20b
gemma3:4b
llama2:latest
llama3:latest


### Explore and Download a Model

Check out and download at least one model from [the list of available models](https://ollama.com/library?sort=popular). Note: if you do not have a GPU or M1+ chip, you probably should start with the smaller models (i.e. 4b or less).

In [10]:
model_name = 'gemma3:4b'

In [11]:
client.pull(model_name)

ProgressResponse(status='success', completed=None, total=None, digest=None)

### Prompt a Model

Try it out!

In [12]:
test_prompt = "ummmm... anyone there?"

In [13]:
messages = [{
    'role': 'user',
    'content': test_prompt
}]

In [14]:
response = client.chat(
    model=model_name,
    messages=messages
)

In [15]:
response

ChatResponse(model='gemma3:4b', created_at='2026-04-23T13:52:28.406892Z', done=True, done_reason='stop', total_duration=5485703542, load_duration=3109291125, prompt_eval_count=15, prompt_eval_duration=726913416, eval_count=46, eval_duration=1591387836, message=Message(role='assistant', content="Yes, hello there! I'm here. 😊 \n\nIt's nice to connect with you. How can I help you today? Do you have a question, need some information, or just want to chat?", thinking=None, images=None, tool_name=None, tool_calls=None))

In [16]:
response.__dict__

{'model': 'gemma3:4b',
 'created_at': '2026-04-23T13:52:28.406892Z',
 'done': True,
 'done_reason': 'stop',
 'total_duration': 5485703542,
 'load_duration': 3109291125,
 'prompt_eval_count': 15,
 'prompt_eval_duration': 726913416,
 'eval_count': 46,
 'eval_duration': 1591387836,
 'message': Message(role='assistant', content="Yes, hello there! I'm here. 😊 \n\nIt's nice to connect with you. How can I help you today? Do you have a question, need some information, or just want to chat?", thinking=None, images=None, tool_name=None, tool_calls=None)}

In [17]:
print(response.message.content)

Yes, hello there! I'm here. 😊 

It's nice to connect with you. How can I help you today? Do you have a question, need some information, or just want to chat?


In [ ]:
del client

### Let's get llamafile going

1. Follow the [llamafile quickstart](https://mozilla-ai.github.io/llamafile/quickstart/).
2. Complete steps 1-4 in the llamafile quickstart.
   You will see the llamafile chat in the terminal.
3. Navigate to http://localhost:8080/ to open llama.cpp in your browser.
   It will have lots of options to enter a prompt. You can test it in the browser at the bottom of the page that opened.

🛑 **Do the steps above before continuing.**

In [ ]:
from openai import OpenAI

In [ ]:
client = OpenAI(
    base_url="http://localhost:8080/v1", # "http://<Your api-server IP>:port"
    api_key = "sk-no-key-required"
)

In [ ]:
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=messages
)

In [ ]:
completion

In [ ]:
completion.__dict__

In [ ]:
completion.choices[0].message.content

Well, we can see not all models are built alike ;)

### Building some ways to load templates and save chats

We will set up:

1. a very simple meta prompt template
2. an example prompt
3. an example conversation

The point is to give us some conventions around saving our work as we progress and building best practices to save conversations for later evaluation and review.

In [ ]:
import json

In [ ]:
meta_prompt_template = """
{role_description}

{task_description}

{rules}

{examples}

{context}
"""


In [ ]:
role = "You are a helpful assistant who makes tasty recipes."
task = """You use regular pantry items and things everyone has 
in their kitchen and help them make new, inventive recipes with those ingredients."""
rules = "You do not use fancy ingredients that people might not have access to use."
examples = "An example recipe would be a French omlette with spruced up canned beans and a side salad with a quirky vinegrette."
context = "You take inspiration from Nigella Lawson."

In [ ]:
meta_prompt_template.format(
    role_description=role,
    task_description=task,
    rules=rules,
    examples=examples,
    context=context)

In [ ]:
with open('data/templates/meta_prompt_template', 'w') as mpt:
    mpt.write(meta_prompt_template)

In [ ]:
with open('data/meta_prompts/pantry_chef_prompt', 'w') as cpt:
    cpt.write(meta_prompt_template.format(
        role_description=role,
        task_description=task,
        rules=rules,
        examples=examples,
        context=context))

In [ ]:
meta_prompt = meta_prompt_template.format(
    role_description=role,
    task_description=task,
    rules=rules,
    examples=examples,
    context=context)

In [ ]:
conversation = [
    {'role': 'system', 
     'content': meta_prompt},
    {'role': 'user',
     'content': """Help! I only have canned tomatoes, some chicken and an onion.... 
     what can I make for dinner?"""},
]

In [ ]:
completion = client.chat.completions.create(
    model="LLaMA_CPP",
    messages=conversation
)

In [ ]:
completion

In [ ]:
completion.choices[0].message.content

In [ ]:
conversation.append({
    'role': 'agent',
    'content': completion.choices[0].message.content
})

In [ ]:
with open('data/traces/chef_chicken_pasta.json', 'w') as chef_chat:
    json.dump(conversation, chef_chat)

In [ ]:
from pprint import pprint

In [ ]:
pprint(conversation)